# batchnorm-running-stats — worked example 3: Reproduce eval-mode BN by hand from the running buffers

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-running-stats`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Eval-mode BatchNorm is just a fixed per-channel affine transform: `y = weight * (x - running_mean) / sqrt(running_var + eps) + bias`. Knowing this lets you reconstruct the exact module output from the four stored tensors (`running_mean`, `running_var`, `weight`, `bias`) plus `eps`, without calling the module. The buffers are broadcast across the batch and spatial dimensions, one value per channel.

## Worked solution

**Goal.** Manually compute what `bn(x)` returns in eval mode and confirm it matches the real module.

**Step 1 — grab the channel-wise parameters.** A `BatchNorm2d` exposes `running_mean`, `running_var` (buffers) and `weight`, `bias` (learned affine), each of shape `(C,)`. We also read `bn.eps`, the numerical-stability term added inside the square root.

**Step 2 — reshape for broadcasting.** Input `x` is `(B, C, H, W)`; the stats are `(C,)`. We reshape each stat to `(1, C, 1, 1)` so it broadcasts against the channel axis only. Using einops `rearrange(running_mean, 'c -> 1 c 1 1')` makes the intent explicit.

**Step 3 — standardize then affine.** Compute `(x - mean) / sqrt(var + eps)`, then multiply by `weight` and add `bias`. This is the entire eval forward pass — there is no batch reduction because eval ignores the batch.

**Step 4 — verify.** We put the module in eval mode and call `bn(x)`, then compare to our hand-built tensor with `t.allclose`. They match because we replicated the formula PyTorch uses exactly, including the biased-vs-unbiased question being moot here (eval reads the stored buffer directly, no recomputation).

In [ ]:
bn = t.nn.BatchNorm2d(4)
with t.no_grad():
    bn.running_mean.copy_(t.tensor([0.0, 1.0, -2.0, 0.5]))
    bn.running_var.copy_(t.tensor([1.0, 4.0, 0.25, 2.0]))
    bn.weight.copy_(t.tensor([1.0, 2.0, 0.5, 1.5]))
    bn.bias.copy_(t.tensor([0.0, -1.0, 1.0, 0.0]))
bn.eval()

t.manual_seed(0)
x = t.randn(2, 4, 3, 3)

mean = rearrange(bn.running_mean, 'c -> 1 c 1 1')
var = rearrange(bn.running_var, 'c -> 1 c 1 1')
w = rearrange(bn.weight, 'c -> 1 c 1 1')
b = rearrange(bn.bias, 'c -> 1 c 1 1')
manual = w * (x - mean) / t.sqrt(var + bn.eps) + b

reference = bn(x)
print("manual matches module:", t.allclose(manual, reference, atol=1e-6))
print("sample manual[0,1,0,0]:", round(manual[0, 1, 0, 0].item(), 4))